# End-to-end Black–Scholes implied-volatility-function baseline

This notebook builds a reproducible **SPX Black–Scholes / implied-volatility-function (BS–IVF) baseline** from raw OptionMetrics IvyDB US CSV files.

It covers:

1. loading and auditing the raw files;
2. cleaning each table separately;
3. joining option quotes to spot, forward and zero-curve data;
4. quote-quality and no-arbitrage filters;
5. filling missing implied volatilities by inverting cleaned mid-prices;
6. fitting a daily Dumas-style ad hoc implied-volatility function;
7. same-day pricing diagnostics;
8. one-trading-day-ahead pricing;
9. one-day discrete delta hedging;
10. exporting coefficients, cleaning reports and performance summaries.

The Dumas-style ad hoc model uses forward moneyness

$$
m=\frac{K}{F}-1
$$

and switches specification according to the number of expiries in a daily cross-section:

\begin{aligned}
\text{AH1: }\sigma &= \beta_0+\beta_1m+\beta_2m^2,\\
\text{AH2: }\sigma &= \beta_0+\beta_1m+\beta_2m^2+\beta_3T+\beta_4mT,\\
\text{AH3: }\sigma &= \beta_0+\beta_1m+\beta_2m^2+\beta_3T+\beta_4T^2+\beta_5mT.
\end{aligned}


A flat-volatility Black–Scholes model is retained as a simpler benchmark.

> **Scope:** this is the ad hoc BS–IVF baseline discussed by Dumas, Fleming and Whaley, not their deterministic/local-volatility PDE model.

## 0. Before running

Put this notebook in the same folder as the five CSV files:

- `one_month_spx_option_price.csv`
- `one_month_spx_forward_price.csv`
- `one_month_spx_zero_coupon_yield_curve.csv`
- `one_month_spx_security_price.csv`
- `spx_security_name_history.csv`

The code can also find files with suffixes such as `(1)` or `(3)`.

Required packages:

```text
pandas
numpy
scipy
matplotlib
```

Install a missing package with, for example:

```python
%pip install pandas numpy scipy matplotlib
```

In [ ]:
from __future__ import annotations

from collections import OrderedDict
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("Imports completed.")

## 1. Configuration

These settings are deliberately collected in one place.

The two assumptions that should later be checked against the IvyDB variable documentation are:

- zero-curve rates are stored in percentage points, so `5.28` means `5.28%`;
- the zero rates are treated as continuously compounded.

Both assumptions are explicit below and can be changed without rewriting the notebook.

In [ ]:
# ---------- Files ----------
DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR / "outputs_bs_ivf"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_FILES = {
    "option": "one_month_spx_option_price.csv",
    "forward": "one_month_spx_forward_price.csv",
    "zero_curve": "one_month_spx_zero_coupon_yield_curve.csv",
    "security_price": "one_month_spx_security_price.csv",
    "security_name": "spx_security_name_history.csv",
}

FILE_PATTERNS = {
    "option": "*spx*option*price*.csv",
    "forward": "*spx*forward*price*.csv",
    "zero_curve": "*spx*zero*coupon*yield*curve*.csv",
    "security_price": "*spx*security*price*.csv",
    "security_name": "*spx*security*name*history*.csv",
}

# ---------- Sample ----------
TARGET_SECID = 108105
DAY_COUNT = 365.0

# Excluding the very shortest expiries avoids 0DTE microstructure effects
# and keeps option maturities inside the downloaded zero-curve range.
MIN_DTE = 14
MAX_DTE = 365

# Dumas moneyness m = K/F - 1.
MAX_ABS_MONEYNESS = 0.30

# ---------- Quote-quality filters ----------
REQUIRE_POSITIVE_BID = True
MIN_MID_PRICE = 0.50
MAX_RELATIVE_SPREAD = 0.50
REQUIRE_EUROPEAN = True
REQUIRE_STANDARD_CONTRACT = True
STANDARD_CONTRACT_SIZE = 100.0

# Fit and evaluation generally use OTM options because their implied
# volatilities are more stable than those of deeply ITM options.
FIT_OTM_ONLY = True
EVALUATE_OTM_ONLY = True

# ---------- Implied volatility ----------
OPTION_IV_INPUT_IS_PERCENT = False
RECOMPUTE_ALL_IV = False
CHECK_VENDOR_IV_SAMPLE = 500
IV_LOWER_BOUND = 0.005
IV_UPPER_BOUND = 5.0
IVF_OUTPUT_FLOOR = 0.01
IVF_OUTPUT_CAP = 3.00

# ---------- Daily regression ----------
MIN_FIT_OBSERVATIONS = 30
MIN_OBSERVATIONS_PER_PARAMETER = 5
FIT_WEIGHTING = "ols"  # "ols" is closest to a simple ad hoc IV smoother.
                       # "vega" is available as a robustness check.

# ---------- Interest rates ----------
ZERO_RATE_INPUT_IS_PERCENT = True
ZERO_RATE_COMPOUNDING = "continuous"  # alternative supported value: "annual"

# ---------- Outputs ----------
SAVE_LARGE_ROW_LEVEL_CSVS = True

print("Configuration loaded.")

## 2. Resolve and load the CSV files

The resolver first looks for the preferred filename. If it is absent, it searches for matching CSV files and selects the most recently modified one. The chosen path is printed, so check it before continuing.

In [ ]:
def resolve_file(label: str) -> Path:
    preferred = DATA_DIR / PREFERRED_FILES[label]
    if preferred.exists():
        print(f"{label:>15}: {preferred}")
        return preferred

    candidates = sorted(
        DATA_DIR.glob(FILE_PATTERNS[label]),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"Could not find the {label!r} CSV in {DATA_DIR.resolve()}. "
            f"Expected {preferred.name!r} or pattern {FILE_PATTERNS[label]!r}."
        )

    chosen = candidates[0]
    print(f"{label:>15}: {chosen}  [selected from {len(candidates)} match(es)]")
    return chosen


PATHS = {label: resolve_file(label) for label in PREFERRED_FILES}

option_raw = pd.read_csv(PATHS["option"], low_memory=False)
forward_raw = pd.read_csv(PATHS["forward"], low_memory=False)
zero_curve_raw = pd.read_csv(PATHS["zero_curve"], low_memory=False)
security_price_raw = pd.read_csv(PATHS["security_price"], low_memory=False)
security_name_raw = pd.read_csv(PATHS["security_name"], low_memory=False)

print("\nRaw files loaded.")

## 3. Raw-data audit

This section runs **before cleaning**. It makes data problems visible rather than silently deleting them.

In [ ]:
def normalise_column_names(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result.columns = (
        pd.Index(result.columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return result


def audit_frame(name: str, frame: pd.DataFrame, key: list[str] | None = None) -> None:
    print("=" * 90)
    print(name)
    print("=" * 90)
    print(f"Shape: {frame.shape}")
    print(f"Columns: {frame.columns.tolist()}")

    missing = frame.isna().mean().mul(100).sort_values(ascending=False)
    print("\nMissing percentages (nonzero only):")
    print(missing[missing > 0].round(3).to_string() or "None")

    if key is not None and all(column in frame.columns for column in key):
        print(f"\nDuplicate rows on {key}: {frame.duplicated(key).sum():,}")

    print("\nFirst three rows:")
    print(frame.head(3).to_string(index=False))
    print()


def audit_option_prices(frame: pd.DataFrame) -> None:
    opt = normalise_column_names(frame)

    print("=" * 90)
    print("OPTION PRICES: EXTRA AUDIT")
    print("=" * 90)
    print(f"Shape: {opt.shape}")

    for column in ["date", "exdate"]:
        if column in opt:
            parsed = pd.to_datetime(opt[column], errors="coerce")
            print(
                f"{column}: {parsed.min()} to {parsed.max()}, "
                f"{parsed.nunique()} unique values, {parsed.isna().sum():,} unparseable"
            )

    print("\nImportant flag values:")
    for column in [
        "secid", "cp_flag", "exercise_style", "ss_flag",
        "am_settlement", "contract_size", "expiry_indicator", "index_flag"
    ]:
        if column in opt:
            print(f"\n{column}:")
            print(opt[column].value_counts(dropna=False).head(20).to_string())

    if {"optionid", "date"}.issubset(opt.columns):
        print(
            "\nDuplicate (optionid, date):",
            f"{opt.duplicated(['optionid', 'date']).sum():,}",
        )

    numeric = {}
    for column in [
        "strike_price", "best_bid", "best_offer", "volume", "open_interest",
        "impl_volatility", "delta", "gamma", "vega", "theta"
    ]:
        if column in opt:
            numeric[column] = pd.to_numeric(opt[column], errors="coerce")

    if {"best_bid", "best_offer"}.issubset(numeric):
        bid = numeric["best_bid"]
        ask = numeric["best_offer"]
        print("\nBasic quote issues:")
        print(f"missing bid:       {bid.isna().sum():,}")
        print(f"missing offer:     {ask.isna().sum():,}")
        print(f"negative bid:      {(bid < 0).sum():,}")
        print(f"nonpositive offer: {(ask <= 0).sum():,}")
        print(f"bid above offer:   {(bid > ask).sum():,}")
        print(f"zero bid:          {(bid == 0).sum():,}")

    if "impl_volatility" in numeric:
        iv = numeric["impl_volatility"]
        print(f"missing supplied IV: {iv.isna().sum():,} ({iv.isna().mean():.2%})")

    print()


audit_frame(
    "FORWARD PRICES",
    normalise_column_names(forward_raw),
)
audit_frame(
    "ZERO-COUPON CURVE",
    normalise_column_names(zero_curve_raw),
    key=["date", "days"],
)
audit_frame(
    "SECURITY PRICES",
    normalise_column_names(security_price_raw),
    key=["secid", "date"],
)
audit_frame(
    "SECURITY NAME HISTORY",
    normalise_column_names(security_name_raw),
)
audit_option_prices(option_raw)

## 4. Clean the four supporting tables

Each table is cleaned and validated **before** it is merged into the option table.

The new forward-price file should contain `AMSettlement`. This resolves the duplicate third-Friday forward records by making

```text
(secid, date, expiration, AMSettlement)
```

the complete key.

In [ ]:
def require_columns(frame: pd.DataFrame, required: list[str], name: str) -> None:
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")


def clean_security_prices(frame: pd.DataFrame) -> pd.DataFrame:
    result = normalise_column_names(frame)
    require_columns(result, ["secid", "date", "close"], "Security Prices")

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    for column in ["secid", "low", "high", "open", "close", "return"]:
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="coerce")

    result = result.loc[result["secid"].eq(TARGET_SECID)].copy()

    if result.duplicated(["secid", "date"]).any():
        duplicates = result.loc[result.duplicated(["secid", "date"], keep=False)]
        raise ValueError(
            "Security Prices is not unique on (secid, date).\n"
            + duplicates.head(10).to_string(index=False)
        )

    return result.sort_values(["secid", "date"]).reset_index(drop=True)


def clean_forward_prices(frame: pd.DataFrame) -> pd.DataFrame:
    result = normalise_column_names(frame)
    result = result.rename(
        columns={
            "forwardprice": "forward_price",
            "amsettlement": "am_settlement",
            "expiration": "exdate",
        }
    )
    require_columns(
        result,
        ["secid", "date", "exdate", "am_settlement", "forward_price"],
        "Forward Prices",
    )

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["exdate"] = pd.to_datetime(result["exdate"], errors="coerce")
    result["secid"] = pd.to_numeric(result["secid"], errors="coerce")
    result["am_settlement"] = pd.to_numeric(
        result["am_settlement"], errors="coerce"
    ).astype("Int64")
    result["forward_price"] = pd.to_numeric(
        result["forward_price"], errors="coerce"
    )

    result = result.loc[result["secid"].eq(TARGET_SECID)].copy()

    key = ["secid", "date", "exdate", "am_settlement"]
    if result.duplicated(key).any():
        duplicates = result.loc[result.duplicated(key, keep=False)]
        raise ValueError(
            f"Forward Prices is not unique on {key}.\n"
            + duplicates.head(10).to_string(index=False)
        )

    return result.sort_values(key).reset_index(drop=True)


def clean_zero_curve(frame: pd.DataFrame) -> pd.DataFrame:
    result = normalise_column_names(frame)
    require_columns(result, ["date", "days", "rate"], "Zero Curve")

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["days"] = pd.to_numeric(result["days"], errors="coerce")
    result["rate"] = pd.to_numeric(result["rate"], errors="coerce")

    if ZERO_RATE_INPUT_IS_PERCENT:
        result["zero_rate"] = result["rate"] / 100.0
    else:
        result["zero_rate"] = result["rate"]

    result = result.dropna(subset=["date", "days", "zero_rate"]).copy()

    if result.duplicated(["date", "days"]).any():
        duplicates = result.loc[result.duplicated(["date", "days"], keep=False)]
        raise ValueError(
            "Zero curve is not unique on (date, days).\n"
            + duplicates.head(10).to_string(index=False)
        )

    return result.sort_values(["date", "days"]).reset_index(drop=True)


def clean_security_name_history(frame: pd.DataFrame) -> pd.DataFrame:
    result = normalise_column_names(frame)
    require_columns(result, ["secid", "effect_date", "ticker", "issuer"], "Security Name History")

    result["secid"] = pd.to_numeric(result["secid"], errors="coerce")
    result["effect_date"] = pd.to_datetime(result["effect_date"], errors="coerce")
    return (
        result.loc[result["secid"].eq(TARGET_SECID)]
        .sort_values(["secid", "effect_date"])
        .reset_index(drop=True)
    )


security_prices = clean_security_prices(security_price_raw)
forward_prices = clean_forward_prices(forward_raw)
zero_curve = clean_zero_curve(zero_curve_raw)
security_names = clean_security_name_history(security_name_raw)

print("Clean supporting-table shapes:")
print("security_prices:", security_prices.shape)
print("forward_prices: ", forward_prices.shape)
print("zero_curve:     ", zero_curve.shape)
print("security_names: ", security_names.shape)

print("\nForward key duplicates after including AM settlement:")
print(
    forward_prices.duplicated(
        ["secid", "date", "exdate", "am_settlement"]
    ).sum()
)

print("\nSPX identity check:")
print(security_names.tail().to_string(index=False))

## 5. Basic option cleaning

At this stage we only:

- standardise types;
- parse dates;
- convert `strike_price / 1000`;
- construct bid–ask midpoints and spreads;
- retain the original OptionMetrics IV and Greeks for later comparison.

No research-sample filtering is applied yet.

In [ ]:
def prepare_option_prices(frame: pd.DataFrame) -> pd.DataFrame:
    result = normalise_column_names(frame)

    required = [
        "secid", "date", "exdate", "cp_flag", "strike_price",
        "best_bid", "best_offer", "optionid", "am_settlement",
        "contract_size", "ss_flag", "exercise_style"
    ]
    require_columns(result, required, "Option Prices")

    # Avoid a name collision with the separate Forward Price table.
    if "forward_price" in result.columns:
        result = result.rename(
            columns={"forward_price": "forward_price_in_option_file"}
        )

    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    result["exdate"] = pd.to_datetime(result["exdate"], errors="coerce")

    numeric_columns = [
        "secid", "strike_price", "best_bid", "best_offer", "volume",
        "open_interest", "impl_volatility", "delta", "gamma", "vega",
        "theta", "optionid", "cfadj", "am_settlement", "contract_size",
        "forward_price_in_option_file", "index_flag"
    ]
    for column in numeric_columns:
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="coerce")

    result["am_settlement"] = result["am_settlement"].astype("Int64")
    result["cp_flag"] = result["cp_flag"].astype("string").str.strip().str.upper()
    result["exercise_style"] = (
        result["exercise_style"].astype("string").str.strip().str.upper()
    )
    result["ss_flag_normalised"] = (
        result["ss_flag"].astype("string").str.strip().str.upper()
    )

    result["strike"] = result["strike_price"] / 1000.0
    result["mid"] = (result["best_bid"] + result["best_offer"]) / 2.0
    result["spread"] = result["best_offer"] - result["best_bid"]
    result["relative_spread"] = result["spread"] / result["mid"].where(
        result["mid"] > 0
    )
    result["dte"] = (result["exdate"] - result["date"]).dt.days
    result["time_to_expiry"] = result["dte"] / DAY_COUNT

    return result


options_prepared = prepare_option_prices(option_raw)

print("Prepared option shape:", options_prepared.shape)
print("\nDate range:", options_prepared["date"].min(), "to", options_prepared["date"].max())
print("\nStrike check:")
print(options_prepared["strike"].describe().to_string())

## 6. Merge spot and forward prices

Both joins are left joins from Option Prices. This lets us count missing support data instead of silently deleting unmatched option rows.

In [ ]:
options_merged = options_prepared.merge(
    security_prices[["secid", "date", "close"]].rename(
        columns={"close": "spot"}
    ),
    on=["secid", "date"],
    how="left",
    validate="many_to_one",
)

options_merged = options_merged.merge(
    forward_prices[
        ["secid", "date", "exdate", "am_settlement", "forward_price"]
    ],
    on=["secid", "date", "exdate", "am_settlement"],
    how="left",
    validate="many_to_one",
)

print("Rows after support-table merges:", len(options_merged))
print(f"Missing spot:    {options_merged['spot'].isna().sum():,}")
print(f"Missing forward: {options_merged['forward_price'].isna().sum():,}")

forward_match_by_settlement = (
    options_merged.assign(matched=options_merged["forward_price"].notna())
    .groupby("am_settlement", dropna=False)["matched"]
    .agg(["count", "mean"])
)
forward_match_by_settlement["mean"] *= 100
forward_match_by_settlement = forward_match_by_settlement.rename(
    columns={"mean": "matched_percent"}
)
forward_match_by_settlement

## 7. Interpolate maturity-matched zero rates

For option valuation, the interpolation does **not** extrapolate outside the downloaded curve. Rows outside the curve are flagged and later excluded.

For overnight hedge financing, the shortest curve rate is used as a flat short-end approximation because the downloaded curve begins at 10 days. This is explicit rather than hidden.

In [ ]:
def interpolate_curve_rates(
    frame: pd.DataFrame,
    curve: pd.DataFrame,
    target_days_column: str,
    *,
    allow_flat_extrapolation: bool,
) -> pd.Series:
    output = pd.Series(np.nan, index=frame.index, dtype=float)
    curve_groups = {
        date: group.sort_values("days")
        for date, group in curve.groupby("date", sort=False)
    }

    for date, indices in frame.groupby("date", sort=False).groups.items():
        daily_curve = curve_groups.get(date)
        if daily_curve is None or daily_curve.empty:
            continue

        x = daily_curve["days"].to_numpy(dtype=float)
        y = daily_curve["zero_rate"].to_numpy(dtype=float)
        targets = frame.loc[indices, target_days_column].to_numpy(dtype=float)

        interpolated = np.interp(targets, x, y)

        if not allow_flat_extrapolation:
            outside = (targets < x.min()) | (targets > x.max())
            interpolated[outside] = np.nan

        output.loc[indices] = interpolated

    return output


def discount_factor(rate, maturity):
    rate = np.asarray(rate, dtype=float)
    maturity = np.asarray(maturity, dtype=float)

    if ZERO_RATE_COMPOUNDING == "continuous":
        return np.exp(-rate * maturity)
    if ZERO_RATE_COMPOUNDING == "annual":
        return np.power(1.0 + rate, -maturity)

    raise ValueError(
        "ZERO_RATE_COMPOUNDING must be 'continuous' or 'annual'."
    )


options_merged["zero_rate"] = interpolate_curve_rates(
    options_merged,
    zero_curve,
    "dte",
    allow_flat_extrapolation=False,
)
options_merged["discount_factor"] = discount_factor(
    options_merged["zero_rate"],
    options_merged["time_to_expiry"],
)

options_merged["moneyness"] = (
    options_merged["strike"] / options_merged["forward_price"] - 1.0
)
options_merged["log_moneyness"] = np.log(
    options_merged["strike"] / options_merged["forward_price"]
)

is_call = options_merged["cp_flag"].eq("C")
is_put = options_merged["cp_flag"].eq("P")

options_merged["is_otm"] = (
    (is_call & options_merged["strike"].ge(options_merged["forward_price"]))
    | (is_put & options_merged["strike"].lt(options_merged["forward_price"]))
)

# European forward-price no-arbitrage bounds.
options_merged["arbitrage_lower_bound"] = np.where(
    is_call,
    options_merged["discount_factor"]
    * np.maximum(options_merged["forward_price"] - options_merged["strike"], 0.0),
    options_merged["discount_factor"]
    * np.maximum(options_merged["strike"] - options_merged["forward_price"], 0.0),
)
options_merged["arbitrage_upper_bound"] = np.where(
    is_call,
    options_merged["discount_factor"] * options_merged["forward_price"],
    options_merged["discount_factor"] * options_merged["strike"],
)

print(f"Missing interpolated option rate: {options_merged['zero_rate'].isna().sum():,}")
print(
    "Zero-rate range used:",
    options_merged["zero_rate"].min(),
    "to",
    options_merged["zero_rate"].max(),
)

## 8. Structural and quote-quality filters

Every filter is applied sequentially and logged. Mean imputation and forward filling are not used.

The report records:

- rows entering a step;
- rows removed by that step;
- rows remaining.

In [ ]:
def is_standard_ss_flag(series: pd.Series) -> pd.Series:
    return series.isin(["0", "0.0"])


def apply_sequential_filters(
    frame: pd.DataFrame,
    conditions: OrderedDict[str, pd.Series],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    keep = pd.Series(True, index=frame.index)
    records = []

    for name, condition in conditions.items():
        condition = condition.fillna(False)
        before = int(keep.sum())
        newly_removed = keep & ~condition
        keep &= condition
        after = int(keep.sum())

        records.append(
            {
                "filter": name,
                "rows_before": before,
                "rows_removed": int(newly_removed.sum()),
                "rows_after": after,
                "percent_of_original_remaining": 100.0 * after / len(frame),
            }
        )

    return frame.loc[keep].copy(), pd.DataFrame(records)


valid_quote = (
    options_merged["best_bid"].notna()
    & options_merged["best_offer"].notna()
    & options_merged["best_bid"].ge(0)
    & options_merged["best_offer"].gt(0)
    & options_merged["best_offer"].ge(options_merged["best_bid"])
    & options_merged["mid"].gt(0)
)

strictly_inside_bounds = (
    options_merged["mid"].gt(options_merged["arbitrage_lower_bound"] + 1e-10)
    & options_merged["mid"].lt(options_merged["arbitrage_upper_bound"] - 1e-10)
)

conditions = OrderedDict(
    [
        ("target secid", options_merged["secid"].eq(TARGET_SECID)),
        (
            "valid identifiers and dates",
            options_merged[
                ["date", "exdate", "optionid", "strike"]
            ].notna().all(axis=1)
            & options_merged["strike"].gt(0)
            & options_merged["cp_flag"].isin(["C", "P"]),
        ),
        (
            "spot, forward and rate matched",
            options_merged[
                ["spot", "forward_price", "zero_rate", "discount_factor"]
            ].notna().all(axis=1)
            & options_merged["spot"].gt(0)
            & options_merged["forward_price"].gt(0),
        ),
        (
            "European exercise",
            options_merged["exercise_style"].eq("E")
            if REQUIRE_EUROPEAN
            else pd.Series(True, index=options_merged.index),
        ),
        (
            "standard contract flags",
            (
                is_standard_ss_flag(options_merged["ss_flag_normalised"])
                & options_merged["contract_size"].eq(STANDARD_CONTRACT_SIZE)
            )
            if REQUIRE_STANDARD_CONTRACT
            else pd.Series(True, index=options_merged.index),
        ),
        ("valid two-sided quote", valid_quote),
        (
            "positive bid",
            options_merged["best_bid"].gt(0)
            if REQUIRE_POSITIVE_BID
            else pd.Series(True, index=options_merged.index),
        ),
        ("minimum mid-price", options_merged["mid"].ge(MIN_MID_PRICE)),
        (
            "relative spread limit",
            options_merged["relative_spread"].le(MAX_RELATIVE_SPREAD),
        ),
        (
            "days-to-expiry range",
            options_merged["dte"].between(MIN_DTE, MAX_DTE),
        ),
        (
            "moneyness range",
            options_merged["moneyness"].abs().le(MAX_ABS_MONEYNESS),
        ),
        ("strict no-arbitrage bounds", strictly_inside_bounds),
    ]
)

options_structural, cleaning_report = apply_sequential_filters(
    options_merged,
    conditions,
)

print("Rows before filtering:", f"{len(options_merged):,}")
print("Rows after structural filtering:", f"{len(options_structural):,}")
cleaning_report

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(cleaning_report["filter"], cleaning_report["rows_removed"])
ax.set_title("Rows removed at each cleaning step")
ax.set_ylabel("Rows removed")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()

## 9. Black–Scholes functions in forward form

With forward \(F\), strike \(K\), discount factor \(D\), maturity \(T\), and volatility \(\sigma\):

\[
C=D\left(FN(d_1)-KN(d_2)
ight).
\]

The corresponding put formula follows from put–call parity.

The spot delta uses the observed carry ratio \(F/S\):

\[
\Delta_{	ext{call}}
=
D
rac{F}{S}N(d_1).
\]

In [ ]:
def _as_float_array(value):
    return np.asarray(value, dtype=float)


def black_forward_price(cp_flag, forward, strike, maturity, discount, sigma):
    cp = np.asarray(cp_flag)
    forward = _as_float_array(forward)
    strike = _as_float_array(strike)
    maturity = _as_float_array(maturity)
    discount = _as_float_array(discount)
    sigma = _as_float_array(sigma)

    sqrt_t = np.sqrt(maturity)
    denominator = sigma * sqrt_t
    d1 = (
        np.log(forward / strike) + 0.5 * sigma * sigma * maturity
    ) / denominator
    d2 = d1 - denominator

    call = discount * (
        forward * norm.cdf(d1) - strike * norm.cdf(d2)
    )
    put = discount * (
        strike * norm.cdf(-d2) - forward * norm.cdf(-d1)
    )
    return np.where(cp == "C", call, put)


def black_forward_vega(forward, maturity, discount, sigma, strike):
    forward = _as_float_array(forward)
    maturity = _as_float_array(maturity)
    discount = _as_float_array(discount)
    sigma = _as_float_array(sigma)
    strike = _as_float_array(strike)

    sqrt_t = np.sqrt(maturity)
    d1 = (
        np.log(forward / strike) + 0.5 * sigma * sigma * maturity
    ) / (sigma * sqrt_t)
    return discount * forward * norm.pdf(d1) * sqrt_t


def black_spot_delta(
    cp_flag,
    spot,
    forward,
    strike,
    maturity,
    discount,
    sigma,
):
    cp = np.asarray(cp_flag)
    spot = _as_float_array(spot)
    forward = _as_float_array(forward)
    strike = _as_float_array(strike)
    maturity = _as_float_array(maturity)
    discount = _as_float_array(discount)
    sigma = _as_float_array(sigma)

    sqrt_t = np.sqrt(maturity)
    d1 = (
        np.log(forward / strike) + 0.5 * sigma * sigma * maturity
    ) / (sigma * sqrt_t)

    carry_discount = discount * forward / spot
    call_delta = carry_discount * norm.cdf(d1)
    put_delta = carry_discount * (norm.cdf(d1) - 1.0)
    return np.where(cp == "C", call_delta, put_delta)


def implied_volatility_scalar(
    price: float,
    cp_flag: str,
    forward: float,
    strike: float,
    maturity: float,
    discount: float,
) -> float:
    values = [price, forward, strike, maturity, discount]
    if not np.isfinite(values).all():
        return np.nan
    if price <= 0 or forward <= 0 or strike <= 0 or maturity <= 0 or discount <= 0:
        return np.nan

    def objective(sigma: float) -> float:
        model_price = black_forward_price(
            cp_flag,
            forward,
            strike,
            maturity,
            discount,
            sigma,
        )
        return float(model_price - price)

    try:
        low_value = objective(IV_LOWER_BOUND)
        high_value = objective(IV_UPPER_BOUND)

        if low_value == 0:
            return IV_LOWER_BOUND
        if high_value == 0:
            return IV_UPPER_BOUND
        if low_value * high_value > 0:
            return np.nan

        return brentq(
            objective,
            IV_LOWER_BOUND,
            IV_UPPER_BOUND,
            maxiter=100,
            xtol=1e-10,
        )
    except (ValueError, RuntimeError, FloatingPointError):
        return np.nan


print("Black–Scholes functions defined.")

## 10. Construct the market implied volatility

Default behaviour:

1. use a valid OptionMetrics supplied IV;
2. invert the cleaned midpoint only where supplied IV is missing or invalid;
3. independently re-invert a sample of supplied IV rows as a consistency check.

Set `RECOMPUTE_ALL_IV = True` to invert every retained quote. That is slower and is usually unnecessary during initial development.

In [ ]:
options_iv = options_structural.copy()

vendor_iv = pd.to_numeric(
    options_iv["impl_volatility"], errors="coerce"
)

if OPTION_IV_INPUT_IS_PERCENT:
    vendor_iv = vendor_iv / 100.0

vendor_iv_valid = vendor_iv.between(
    IV_LOWER_BOUND, IV_UPPER_BOUND
)

if RECOMPUTE_ALL_IV:
    options_iv["market_iv"] = np.nan
    iv_to_invert = pd.Series(True, index=options_iv.index)
else:
    options_iv["market_iv"] = vendor_iv.where(vendor_iv_valid)
    iv_to_invert = options_iv["market_iv"].isna()

print(
    f"Rows requiring midpoint IV inversion: "
    f"{iv_to_invert.sum():,} of {len(options_iv):,}"
)

for counter, index in enumerate(options_iv.index[iv_to_invert], start=1):
    row = options_iv.loc[index]
    options_iv.at[index, "market_iv"] = implied_volatility_scalar(
        price=float(row["mid"]),
        cp_flag=str(row["cp_flag"]),
        forward=float(row["forward_price"]),
        strike=float(row["strike"]),
        maturity=float(row["time_to_expiry"]),
        discount=float(row["discount_factor"]),
    )

    if counter % 10_000 == 0:
        print(f"Inverted {counter:,} rows...")

options_iv["iv_source"] = np.where(
    vendor_iv_valid & ~RECOMPUTE_ALL_IV,
    "optionmetrics",
    np.where(options_iv["market_iv"].notna(), "midpoint inversion", "failed"),
)

print("\nIV-source counts:")
print(options_iv["iv_source"].value_counts(dropna=False).to_string())

# Independent check of a random sample of vendor IV observations.
vendor_check = options_iv.loc[
    vendor_iv_valid.reindex(options_iv.index, fill_value=False)
].copy()

if not vendor_check.empty and CHECK_VENDOR_IV_SAMPLE > 0:
    vendor_check = vendor_check.sample(
        n=min(CHECK_VENDOR_IV_SAMPLE, len(vendor_check)),
        random_state=123,
    )

    vendor_check["recomputed_iv"] = [
        implied_volatility_scalar(
            price=row.mid,
            cp_flag=row.cp_flag,
            forward=row.forward_price,
            strike=row.strike,
            maturity=row.time_to_expiry,
            discount=row.discount_factor,
        )
        for row in vendor_check.itertuples(index=False)
    ]

    vendor_check["absolute_iv_difference"] = (
        vendor_check["recomputed_iv"] - vendor_check["market_iv"]
    ).abs()

    print("\nVendor-IV sample consistency check:")
    print(
        vendor_check["absolute_iv_difference"]
        .describe(percentiles=[0.5, 0.9, 0.99])
        .to_string()
    )

In [ ]:
iv_conditions = OrderedDict(
    [
        (
            "finite market IV",
            options_iv["market_iv"].notna()
            & np.isfinite(options_iv["market_iv"]),
        ),
        (
            "market IV range",
            options_iv["market_iv"].between(
                IV_LOWER_BOUND, IV_UPPER_BOUND
            ),
        ),
    ]
)

options_clean, iv_cleaning_report = apply_sequential_filters(
    options_iv,
    iv_conditions,
)

cleaning_report_full = pd.concat(
    [
        cleaning_report,
        iv_cleaning_report.assign(
            rows_before=lambda frame: frame["rows_before"],
            rows_after=lambda frame: frame["rows_after"],
        ),
    ],
    ignore_index=True,
)

print("Final clean option rows:", f"{len(options_clean):,}")
print("Unique dates:", options_clean["date"].nunique())
print("Unique contracts:", options_clean["optionid"].nunique())
print("Unique expiries:", options_clean["exdate"].nunique())
print("\nCall/put counts:")
print(options_clean["cp_flag"].value_counts().to_string())
print("\nOTM counts:")
print(options_clean["is_otm"].value_counts().to_string())

## 11. Define and fit the daily IVF models

The fit is performed on one daily cross-section at a time.

- `BS_flat`: one constant IV for the whole cross-section.
- `BS_IVF`: the Dumas-style switching specification AH1/AH2/AH3.

The default fit is ordinary least squares on implied volatility. A vega-weighted robustness check can be selected in the configuration.

In [ ]:
SPEC_FEATURES = {
    "flat": ["const"],
    "ah1": ["const", "m", "m2"],
    "ah2": ["const", "m", "m2", "T", "mT"],
    "ah3": ["const", "m", "m2", "T", "T2", "mT"],
}

COEFFICIENT_COLUMNS = [
    "coef_const",
    "coef_m",
    "coef_m2",
    "coef_T",
    "coef_T2",
    "coef_mT",
]


def choose_dumas_spec(number_of_expiries: int) -> str:
    if number_of_expiries <= 1:
        return "ah1"
    if number_of_expiries == 2:
        return "ah2"
    return "ah3"


def feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame(index=frame.index)
    result["const"] = 1.0
    result["m"] = frame["moneyness"]
    result["m2"] = frame["moneyness"] ** 2
    result["T"] = frame["time_to_expiry"]
    result["T2"] = frame["time_to_expiry"] ** 2
    result["mT"] = frame["moneyness"] * frame["time_to_expiry"]
    return result


def fit_daily_iv_regression(
    frame: pd.DataFrame,
    spec: str,
) -> dict | None:
    features = SPEC_FEATURES[spec]
    X = feature_frame(frame)[features].to_numpy(dtype=float)
    y = frame["market_iv"].to_numpy(dtype=float)

    p = X.shape[1]
    minimum_required = max(
        MIN_FIT_OBSERVATIONS,
        MIN_OBSERVATIONS_PER_PARAMETER * p,
    )
    if len(frame) < minimum_required:
        return None

    if FIT_WEIGHTING == "ols":
        weights = np.ones(len(frame))
    elif FIT_WEIGHTING == "vega":
        weights = black_forward_vega(
            frame["forward_price"],
            frame["time_to_expiry"],
            frame["discount_factor"],
            frame["market_iv"],
            frame["strike"],
        )
        positive = weights[np.isfinite(weights) & (weights > 0)]
        if len(positive) == 0:
            return None
        scale = np.median(positive)
        weights = np.clip(weights / scale, 0.05, 20.0)
    else:
        raise ValueError("FIT_WEIGHTING must be 'ols' or 'vega'.")

    sqrt_w = np.sqrt(weights)
    X_weighted = X * sqrt_w[:, None]
    y_weighted = y * sqrt_w

    beta, _, rank, _ = np.linalg.lstsq(
        X_weighted,
        y_weighted,
        rcond=None,
    )

    fitted = X @ beta
    residual = y - fitted
    sse = float(np.sum(residual ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r_squared = np.nan if sst <= 0 else 1.0 - sse / sst

    coefficient_map = dict(zip(features, beta))
    record = {
        "spec": spec,
        "n_obs": len(frame),
        "n_expiries": frame["exdate"].nunique(),
        "rank": int(rank),
        "condition_number": float(np.linalg.cond(X_weighted)),
        "iv_rmse": float(np.sqrt(np.mean(residual ** 2))),
        "iv_mae": float(np.mean(np.abs(residual))),
        "iv_r_squared": float(r_squared),
    }

    for feature in ["const", "m", "m2", "T", "T2", "mT"]:
        record[f"coef_{feature}"] = coefficient_map.get(feature, 0.0)

    return record


fit_rows = (
    options_clean.loc[options_clean["is_otm"]]
    if FIT_OTM_ONLY
    else options_clean
).copy()

daily_fit_records = []

for date, daily in fit_rows.groupby("date", sort=True):
    n_expiries = daily["exdate"].nunique()

    for model, spec in [
        ("BS_flat", "flat"),
        ("BS_IVF", choose_dumas_spec(n_expiries)),
    ]:
        fitted = fit_daily_iv_regression(daily, spec)
        if fitted is None:
            warnings.warn(
                f"Skipped {model} on {date.date()}: insufficient observations."
            )
            continue

        fitted["date"] = date
        fitted["model"] = model
        daily_fit_records.append(fitted)

daily_coefficients = pd.DataFrame(daily_fit_records).sort_values(
    ["date", "model"]
).reset_index(drop=True)

print("Daily coefficient rows:", len(daily_coefficients))
print("\nModels by date:")
print(
    daily_coefficients.groupby("model")["date"]
    .nunique()
    .to_string()
)
daily_coefficients.head()

## 12. Price and delta under a fitted IVF

Two deltas are calculated:

1. **frozen-IV delta**: plug fitted IV into the ordinary Black–Scholes delta and hold IV fixed;
2. **surface delta**: add the effect of the fitted IV changing when \(F/S\) is held fixed and spot moves.

The frozen-IV delta is the simplest practitioner baseline. Saving both prevents an important modelling choice from being hidden.

In [ ]:
def coefficient_array(frame: pd.DataFrame, name: str) -> np.ndarray:
    return frame[f"coef_{name}"].fillna(0.0).to_numpy(dtype=float)


def fitted_iv_and_derivative(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    m = frame["moneyness"].to_numpy(dtype=float)
    T = frame["time_to_expiry"].to_numpy(dtype=float)

    b0 = coefficient_array(frame, "const")
    b1 = coefficient_array(frame, "m")
    b2 = coefficient_array(frame, "m2")
    bT = coefficient_array(frame, "T")
    bT2 = coefficient_array(frame, "T2")
    bmT = coefficient_array(frame, "mT")

    raw_iv = (
        b0
        + b1 * m
        + b2 * m * m
        + bT * T
        + bT2 * T * T
        + bmT * m * T
    )
    derivative_wrt_m = b1 + 2.0 * b2 * m + bmT * T

    fitted_iv = np.clip(
        raw_iv,
        IVF_OUTPUT_FLOOR,
        IVF_OUTPUT_CAP,
    )

    # Once the floor or cap binds, the clipped function is locally flat.
    clipped = (raw_iv <= IVF_OUTPUT_FLOOR) | (raw_iv >= IVF_OUTPUT_CAP)
    derivative_wrt_m = np.where(clipped, 0.0, derivative_wrt_m)

    return fitted_iv, derivative_wrt_m


def attach_model_values(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    fitted_iv, dsigma_dm = fitted_iv_and_derivative(result)
    result["model_iv"] = fitted_iv

    result["model_price"] = black_forward_price(
        result["cp_flag"],
        result["forward_price"],
        result["strike"],
        result["time_to_expiry"],
        result["discount_factor"],
        result["model_iv"],
    )

    result["delta_frozen_iv"] = black_spot_delta(
        result["cp_flag"],
        result["spot"],
        result["forward_price"],
        result["strike"],
        result["time_to_expiry"],
        result["discount_factor"],
        result["model_iv"],
    )

    vega = black_forward_vega(
        result["forward_price"],
        result["time_to_expiry"],
        result["discount_factor"],
        result["model_iv"],
        result["strike"],
    )

    # m = K/F - 1 and F/S is held fixed locally.
    # dm/dS = -(m + 1)/S.
    dsigma_dspot = (
        dsigma_dm
        * (-(result["moneyness"].to_numpy(dtype=float) + 1.0))
        / result["spot"].to_numpy(dtype=float)
    )

    result["delta_surface"] = (
        result["delta_frozen_iv"] + vega * dsigma_dspot
    )

    return result


evaluation_rows = (
    options_clean.loc[options_clean["is_otm"]]
    if EVALUATE_OTM_ONLY
    else options_clean
).copy()

same_day_parts = []

for date, daily_options in evaluation_rows.groupby("date", sort=True):
    daily_fits = daily_coefficients.loc[
        daily_coefficients["date"].eq(date)
    ]

    for fit in daily_fits.itertuples(index=False):
        part = daily_options.copy()
        part["model"] = fit.model
        part["spec"] = fit.spec

        for coefficient in COEFFICIENT_COLUMNS:
            part[coefficient] = getattr(fit, coefficient)

        part = attach_model_values(part)
        same_day_parts.append(part)

same_day_results = pd.concat(
    same_day_parts,
    ignore_index=True,
) if same_day_parts else pd.DataFrame()

same_day_results["price_error"] = (
    same_day_results["model_price"] - same_day_results["mid"]
)
same_day_results["absolute_price_error"] = (
    same_day_results["price_error"].abs()
)
same_day_results["squared_price_error"] = (
    same_day_results["price_error"] ** 2
)
same_day_results["outside_spread_error"] = np.maximum.reduce(
    [
        same_day_results["best_bid"] - same_day_results["model_price"],
        same_day_results["model_price"] - same_day_results["best_offer"],
        np.zeros(len(same_day_results)),
    ]
)
same_day_results["inside_bid_ask"] = (
    same_day_results["model_price"].between(
        same_day_results["best_bid"],
        same_day_results["best_offer"],
    )
)

print("Same-day model-result rows:", f"{len(same_day_results):,}")
same_day_results[
    [
        "date", "optionid", "cp_flag", "strike", "exdate",
        "market_iv", "model", "model_iv", "mid", "model_price",
        "delta_frozen_iv", "delta_surface"
    ]
].head()

In [ ]:
def price_summary(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    for model, group in frame.groupby("model"):
        records.append(
            {
                "model": model,
                "n": len(group),
                "iv_rmse": np.sqrt(
                    np.mean((group["model_iv"] - group["market_iv"]) ** 2)
                ),
                "price_rmse": np.sqrt(np.mean(group["price_error"] ** 2)),
                "price_mae": np.mean(group["absolute_price_error"]),
                "price_bias": np.mean(group["price_error"]),
                "outside_spread_mae": np.mean(group["outside_spread_error"]),
                "inside_bid_ask_percent": 100.0 * group["inside_bid_ask"].mean(),
            }
        )
    return pd.DataFrame(records).sort_values("model").reset_index(drop=True)


same_day_summary = price_summary(same_day_results)
same_day_summary

## 13. Example daily smile fit

This is a diagnostic plot, not an input to the model. Change `EXAMPLE_DATE` to inspect another day.

In [ ]:
EXAMPLE_DATE = evaluation_rows["date"].sort_values().iloc[0]

example = same_day_results.loc[
    same_day_results["date"].eq(EXAMPLE_DATE)
    & same_day_results["model"].eq("BS_IVF")
].copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    example["moneyness"],
    example["market_iv"],
    s=12,
    alpha=0.45,
    label="Market IV",
)

for exdate, group in example.groupby("exdate"):
    ordered = group.sort_values("moneyness")
    ax.plot(
        ordered["moneyness"],
        ordered["model_iv"],
        linewidth=1,
    )

ax.set_title(f"BS–IVF fit on {EXAMPLE_DATE.date()}")
ax.set_xlabel("Forward moneyness K/F - 1")
ax.set_ylabel("Implied volatility")
ax.legend()
plt.tight_layout()
plt.show()

## 14. Build exact next-trading-day contract pairs

A current observation is paired only with the same `optionid` on the next date appearing in the underlying-price calendar. A contract that reappears after a gap is not incorrectly treated as a one-day observation.

In [ ]:
trading_dates = (
    security_prices["date"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)
next_trading_date = {
    current: following
    for current, following in zip(trading_dates[:-1], trading_dates[1:])
}

current = same_day_results.copy()
current["next_date_expected"] = current["date"].map(next_trading_date)
current = current.loc[current["next_date_expected"].notna()].copy()

next_columns = [
    "optionid", "date", "exdate", "cp_flag", "strike",
    "best_bid", "best_offer", "mid", "spot", "forward_price",
    "zero_rate", "discount_factor", "time_to_expiry",
    "moneyness", "contract_size", "is_otm"
]

next_rows = evaluation_rows[next_columns].copy()
next_rows = next_rows.rename(
    columns={
        column: f"{column}_next"
        for column in next_columns
        if column not in ["optionid"]
    }
)

one_day_pairs = current.merge(
    next_rows,
    left_on=["optionid", "next_date_expected"],
    right_on=["optionid", "date_next"],
    how="inner",
    validate="many_to_one",
)

print("One-day model/contract pairs:", f"{len(one_day_pairs):,}")
print(
    "Unique current dates:",
    one_day_pairs["date"].nunique(),
)
print(
    "Unique option contracts:",
    one_day_pairs["optionid"].nunique(),
)

## 15. Frozen-date-\(t\) one-day-ahead pricing

The coefficients estimated on date \(t\) are frozen. They are evaluated at date \(t+1\) using:

- the new forward;
- the new time to expiry;
- the new discount factor;
- the unchanged strike.

This avoids look-ahead bias.

In [ ]:
def fitted_iv_at_next_state(frame: pd.DataFrame) -> np.ndarray:
    m = frame["moneyness_next"].to_numpy(dtype=float)
    T = frame["time_to_expiry_next"].to_numpy(dtype=float)

    raw = (
        coefficient_array(frame, "const")
        + coefficient_array(frame, "m") * m
        + coefficient_array(frame, "m2") * m * m
        + coefficient_array(frame, "T") * T
        + coefficient_array(frame, "T2") * T * T
        + coefficient_array(frame, "mT") * m * T
    )
    return np.clip(raw, IVF_OUTPUT_FLOOR, IVF_OUTPUT_CAP)


one_day_pairs["predicted_iv_next"] = fitted_iv_at_next_state(
    one_day_pairs
)

one_day_pairs["predicted_price_next"] = black_forward_price(
    one_day_pairs["cp_flag_next"],
    one_day_pairs["forward_price_next"],
    one_day_pairs["strike_next"],
    one_day_pairs["time_to_expiry_next"],
    one_day_pairs["discount_factor_next"],
    one_day_pairs["predicted_iv_next"],
)

one_day_pairs["next_price_error"] = (
    one_day_pairs["predicted_price_next"] - one_day_pairs["mid_next"]
)
one_day_pairs["next_absolute_price_error"] = (
    one_day_pairs["next_price_error"].abs()
)
one_day_pairs["next_outside_spread_error"] = np.maximum.reduce(
    [
        one_day_pairs["best_bid_next"]
        - one_day_pairs["predicted_price_next"],
        one_day_pairs["predicted_price_next"]
        - one_day_pairs["best_offer_next"],
        np.zeros(len(one_day_pairs)),
    ]
)
one_day_pairs["next_inside_bid_ask"] = (
    one_day_pairs["predicted_price_next"].between(
        one_day_pairs["best_bid_next"],
        one_day_pairs["best_offer_next"],
    )
)

one_day_pricing_summary = (
    one_day_pairs.groupby("model")
    .apply(
        lambda group: pd.Series(
            {
                "n": len(group),
                "rmse": np.sqrt(np.mean(group["next_price_error"] ** 2)),
                "mae": np.mean(group["next_absolute_price_error"]),
                "bias": np.mean(group["next_price_error"]),
                "outside_spread_mae": np.mean(
                    group["next_outside_spread_error"]
                ),
                "inside_bid_ask_percent": 100.0
                * group["next_inside_bid_ask"].mean(),
            }
        )
    )
    .reset_index()
)

one_day_pricing_summary

## 16. One-day discrete delta hedge

The notebook evaluates a short-option hedge initiated at the observed midpoint.

At date \(t\):

- sell one option for `mid`;
- buy `delta` units of the index;
- put the remaining cash in the financing account.

At the next trading date, the hedged short-option P&L is

\[
\Pi_{t+1}
=
\Delta_t S_{t+1}
+
(C_t-\Delta_tS_t)e^{r\Delta t}
-
C_{t+1}.
\]

Results are in **index points per option unit**. Multiplying by `contract_size` gives the contract-level result.

> SPX itself is an index rather than a directly traded asset. This notebook therefore treats the hedge as a theoretical position in index units, which is suitable for comparing model deltas. An executable hedge using ES futures or SPY requires a separately documented conversion and basis treatment.

The overnight financing rate is obtained from the date-\(t\) zero curve with flat extrapolation below its shortest maturity.

In [ ]:
one_day_pairs["hedge_horizon_days"] = (
    one_day_pairs["date_next"] - one_day_pairs["date"]
).dt.days
one_day_pairs["hedge_horizon_years"] = (
    one_day_pairs["hedge_horizon_days"] / DAY_COUNT
)

one_day_pairs["cash_rate"] = interpolate_curve_rates(
    one_day_pairs,
    zero_curve,
    "hedge_horizon_days",
    allow_flat_extrapolation=True,
)

if ZERO_RATE_COMPOUNDING == "continuous":
    one_day_pairs["cash_growth"] = np.exp(
        one_day_pairs["cash_rate"]
        * one_day_pairs["hedge_horizon_years"]
    )
elif ZERO_RATE_COMPOUNDING == "annual":
    one_day_pairs["cash_growth"] = np.power(
        1.0 + one_day_pairs["cash_rate"],
        one_day_pairs["hedge_horizon_years"],
    )
else:
    raise ValueError("Unsupported ZERO_RATE_COMPOUNDING value.")


for delta_name in ["delta_frozen_iv", "delta_surface"]:
    suffix = delta_name.removeprefix("delta_")
    delta = one_day_pairs[delta_name]

    initial_cash = (
        one_day_pairs["mid"] - delta * one_day_pairs["spot"]
    )
    hedge_value_next = (
        delta * one_day_pairs["spot_next"]
        + initial_cash * one_day_pairs["cash_growth"]
    )

    pnl = hedge_value_next - one_day_pairs["mid_next"]

    one_day_pairs[f"hedge_pnl_{suffix}"] = pnl
    one_day_pairs[f"hedge_pnl_contract_{suffix}"] = (
        pnl * one_day_pairs["contract_size"]
    )


hedge_long = one_day_pairs.melt(
    id_vars=["model", "date", "date_next", "optionid"],
    value_vars=["hedge_pnl_frozen_iv", "hedge_pnl_surface"],
    var_name="delta_method",
    value_name="hedge_pnl",
)
hedge_long["delta_method"] = (
    hedge_long["delta_method"]
    .str.removeprefix("hedge_pnl_")
)

hedging_summary = (
    hedge_long.groupby(["model", "delta_method"])
    .apply(
        lambda group: pd.Series(
            {
                "n": len(group),
                "pnl_bias": group["hedge_pnl"].mean(),
                "mae": group["hedge_pnl"].abs().mean(),
                "rmse": np.sqrt(np.mean(group["hedge_pnl"] ** 2)),
                "p95_absolute_error": group["hedge_pnl"].abs().quantile(0.95),
            }
        )
    )
    .reset_index()
)

hedging_summary

## 17. Error breakdowns by maturity and moneyness

In [ ]:
one_day_pairs["dte_bucket"] = pd.cut(
    one_day_pairs["dte"],
    bins=[MIN_DTE - 1, 30, 60, 90, 180, MAX_DTE],
    labels=["14–30", "31–60", "61–90", "91–180", "181–365"],
    include_lowest=True,
)

one_day_pairs["moneyness_bucket"] = pd.cut(
    one_day_pairs["moneyness"],
    bins=[-0.30, -0.10, -0.03, 0.03, 0.10, 0.30],
    labels=[
        "deep put wing",
        "put wing",
        "near ATM",
        "call wing",
        "deep call wing",
    ],
    include_lowest=True,
)

pricing_by_dte = (
    one_day_pairs.groupby(["model", "dte_bucket"], observed=True)
    .apply(
        lambda group: pd.Series(
            {
                "n": len(group),
                "rmse": np.sqrt(np.mean(group["next_price_error"] ** 2)),
                "mae": group["next_absolute_price_error"].mean(),
            }
        )
    )
    .reset_index()
)

pricing_by_moneyness = (
    one_day_pairs.groupby(
        ["model", "moneyness_bucket"],
        observed=True,
    )
    .apply(
        lambda group: pd.Series(
            {
                "n": len(group),
                "rmse": np.sqrt(np.mean(group["next_price_error"] ** 2)),
                "mae": group["next_absolute_price_error"].mean(),
            }
        )
    )
    .reset_index()
)

print("Pricing by DTE:")
display(pricing_by_dte)

print("Pricing by moneyness:")
display(pricing_by_moneyness)

## 18. Compact comparison plots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    one_day_pricing_summary["model"],
    one_day_pricing_summary["rmse"],
)
ax.set_title("One-day-ahead pricing RMSE")
ax.set_ylabel("Index points")
plt.tight_layout()
plt.show()

hedge_plot = hedging_summary.copy()
hedge_plot["label"] = (
    hedge_plot["model"] + "\n" + hedge_plot["delta_method"]
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(hedge_plot["label"], hedge_plot["rmse"])
ax.set_title("One-day delta-hedging RMSE")
ax.set_ylabel("Index points")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 19. Export reproducible outputs

The core summaries are always saved. Row-level CSVs can be disabled in the configuration when the full research sample becomes large.

In [ ]:
cleaning_report.to_csv(
    OUTPUT_DIR / "cleaning_report_structural.csv",
    index=False,
)
iv_cleaning_report.to_csv(
    OUTPUT_DIR / "cleaning_report_iv.csv",
    index=False,
)
daily_coefficients.to_csv(
    OUTPUT_DIR / "daily_ivf_coefficients.csv",
    index=False,
)
same_day_summary.to_csv(
    OUTPUT_DIR / "same_day_pricing_summary.csv",
    index=False,
)
one_day_pricing_summary.to_csv(
    OUTPUT_DIR / "one_day_pricing_summary.csv",
    index=False,
)
hedging_summary.to_csv(
    OUTPUT_DIR / "one_day_hedging_summary.csv",
    index=False,
)
pricing_by_dte.to_csv(
    OUTPUT_DIR / "one_day_pricing_by_dte.csv",
    index=False,
)
pricing_by_moneyness.to_csv(
    OUTPUT_DIR / "one_day_pricing_by_moneyness.csv",
    index=False,
)

if SAVE_LARGE_ROW_LEVEL_CSVS:
    same_day_results.to_csv(
        OUTPUT_DIR / "same_day_option_results.csv",
        index=False,
    )
    one_day_pairs.to_csv(
        OUTPUT_DIR / "one_day_option_pairs.csv",
        index=False,
    )

configuration_to_save = {
    name: value
    for name, value in globals().items()
    if name
    in {
        "TARGET_SECID",
        "DAY_COUNT",
        "MIN_DTE",
        "MAX_DTE",
        "MAX_ABS_MONEYNESS",
        "REQUIRE_POSITIVE_BID",
        "MIN_MID_PRICE",
        "MAX_RELATIVE_SPREAD",
        "REQUIRE_EUROPEAN",
        "REQUIRE_STANDARD_CONTRACT",
        "STANDARD_CONTRACT_SIZE",
        "FIT_OTM_ONLY",
        "EVALUATE_OTM_ONLY",
        "OPTION_IV_INPUT_IS_PERCENT",
        "RECOMPUTE_ALL_IV",
        "IV_LOWER_BOUND",
        "IV_UPPER_BOUND",
        "IVF_OUTPUT_FLOOR",
        "IVF_OUTPUT_CAP",
        "MIN_FIT_OBSERVATIONS",
        "MIN_OBSERVATIONS_PER_PARAMETER",
        "FIT_WEIGHTING",
        "ZERO_RATE_INPUT_IS_PERCENT",
        "ZERO_RATE_COMPOUNDING",
    }
}

with open(
    OUTPUT_DIR / "configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(configuration_to_save, file, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

## 20. Interpretation checklist

Before using these results in the UROP:

1. **Confirm the zero-curve convention** in the IvyDB variable documentation.
2. Inspect the forward-price merge rate separately for AM and PM settlement.
3. Examine the cleaning report and do not hide large exclusions.
4. Run sensitivity checks for:
   - 7 versus 14 minimum DTE;
   - maximum relative spread;
   - moneyness range;
   - OLS versus vega weighting;
   - supplied IV versus full midpoint inversion;
   - frozen-IV versus surface delta.
5. Extend beyond one month before drawing empirical conclusions.
6. Use exactly the same cleaned contract-date sample and hedge protocol for BS–IVF, local volatility, Heston and neural SDE comparisons.

The one-month sample is sufficient to debug the complete pipeline. It is not sufficient for final model-performance claims.

## Reference

Dumas, B., Fleming, J. and Whaley, R. E. (1998), *Implied Volatility Functions: Empirical Tests*, Journal of Finance 53(6), 2059–2106.